# Data Exploration Notebook for Approval Predict

## Objectives

Answer business requirement 1:
The client is interested in determining which applicant variables are most strongly correlated with the loan approval outcome. They want a ranked list of variables to be provided based on their relevance and impact.

## Inputs

* outputs/datasets/collection/loan_approval.csv

## Outputs

* generate code that answers business requirement 1 and can be used to build the Streamlit App.

## Imports

In [1]:
import os
from pathlib import Path
import seaborn as sns
from feature_engine.encoding import OneHotEncoder
import matplotlib.pyplot as plt
from feature_engine.discretisation import ArbitraryDiscretiser
import numpy as np
import plotly.express as px
import pandas as pd
import warnings


## Change Working Directory

In [2]:
current_dir = os.getcwd()
current_dir

os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

current_dir = os.getcwd()
current_dir

You set a new current directory


'/Users/davidcarr/Desktop/vscode-projects/Approval_Predict'

## Load Data

In this section, we load the loan approval dataset that was collected and saved in the Data Collection notebook. This dataset contains information about loan applicants including their financial details and the loan approval decision.

We will use this data to perform exploratory analysis and identify which variables are most strongly correlated with loan approval outcomes.

In [3]:
root = current_dir
file_path = Path(root) / "outputs" / "datasets" / "collection" / "loan_approval.csv"

if not file_path.exists():
    raise FileNotFoundError(f"Dataset not found at: {file_path}")

df = pd.read_csv(file_path)
df.head(3)


,name,city,income,credit_score,loan_amount,years_employed,points,loan_approved
0,Allison Hill,East Jill,113810,389,39698,27,50.0,0
1,Brandon Hall,New Jamesside,44592,729,15446,28,55.0,0
2,Rhonda Smith,Lake Roberto,33278,584,11189,13,45.0,0


Data Exploration

In [4]:
from ydata_profiling import ProfileReport
pandas_report = ProfileReport(df=df, minimal=True)
pandas_report.to_notebook_iframe()

ModuleNotFoundError: No module named 'pkg_resources'

In [ ]:
approve_counts = df['loan_approved'].value_counts()
approve_counts

In [ ]:
percentage_approved = df['loan_approved'].value_counts(normalize=True) * 100
percentage_approved

There is a moderate difference imbalance between loan approvals with false having a higher proportion meaning some models may be biased towards the rejection of loan approvals. The variables are split by 5 numbers, text and a boolean.

Convert the target loan_approved and city to numeric values.

In [ ]:
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)
warnings.filterwarnings('ignore')
encoder = OneHotEncoder(variables=df.columns[df.dtypes=='object'].to_list(), drop_last=False)
df_ohe = encoder.fit_transform(df)
print(df_ohe.shape)
df_ohe.head(3)

Sample data with new feature (loan_to_income) to work out loan to income ratio.

In [ ]:
df_ohe['loan_to_income'] = df_ohe['loan_amount'] / df_ohe['income']
df_ohe.head(10)

## Correlation study
This works out pairwise correlation coefficients between all numeric columns in the dataframe.

In [ ]:
df_ohe.dtypes

In [ ]:
corr_spearman = df_ohe.corr(method='spearman')['loan_approved'].sort_values(key=abs, ascending=False)[1:].head(10)
corr_spearman

In [ ]:
corr_pearson = df_ohe.corr(method='pearson')['loan_approved'].sort_values(key=abs, ascending=False)[1:].head(10)
corr_pearson

Spearman correlation measures monotonic relationships, in which variables move in the same direction but not necessarily linear.
Pearson method measures linear relationships, in which 1 is a positive correlation and -1 is a negative correlation.
In the spearman correlation the variable points and credit score have a strong positive correlation suggesting as points increase so does loan approval.
income and loan_amount had a weak correlation and years_employed had a very weak correlation. This would suggest a higher income and lower loan amount would increase loan approval with more year employed slightly improving chances of approval.

For loan_to_income as the ratio increases, approval tends to decrease. Therefore, higher loan-to-income ratios correlate with lower approval but the relationship is quite weak. Higher ratios mean less affordable loans thus increasing rejection.

The city variable has an extremely weak correlation and has little effect on loan approval. 

Pearson correlation showed a similiar pattern where points and credit score had a strong positive correlation and income had a very weak correlation.

Drop feature 'city' as not useful for correlation.

In [ ]:
if 'city' in df.columns:
    df = df.drop(['city'], axis=1)
df.head(4)

In [ ]:
vars_to_study = ['points', 'credit_score', 'income', 'loan_amount', 'years_employed', 'loan_to_income']
vars_to_study

Exploratory Data Analysis (EDA) on selected variables

In [ ]:
df_eda = df.filter(vars_to_study + ['loan_approved'])
df_eda.head(3)

Variables Distribution by loan approved

In [ ]:
%matplotlib inline
sns.set_style('whitegrid')


def plot_numerical(df, vars_to_study, target_var):
    for col in vars_to_study:
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        
        # Histogram
        sns.histplot(
            data=df, x=col, hue=target_var, kde=True, element="step",
            palette='Set1', ax=axes[0]
        )
        axes[0].set_title(f"{col} distribution by {target_var} (Histogram)")
        
        # Boxplot
        sns.boxplot(
            data=df, x=target_var, y=col, palette='Set2', width=0.4, ax=axes[1]
        )
        axes[1].set_title(f"{col} distribution by {target_var} (Boxplot)")
        
        plt.tight_layout()
        plt.show()


In [ ]:
target_var = 'loan_approved'
df['loan_to_income'] = df['loan_amount'] / df['income']


plot_numerical(df, vars_to_study, target_var)

### Points distribution by loan approved summary

The boxplot shows applicants with approved loans have significantly higher points compared to those with rejected loans.

The median points for approved loans is around 75, while for not approved it’s around 45. Therefore, higher points are strongly associated with higher chances of loan approval.

The histogram suggests applicants with higher points are much more likely to get loan approval. Those with lower points of <60 are mostly rejected, while applicants with points above ~70 are primarily approved.

### Credit score distribution by loan approved

The boxplot shows approved applicants have much higher credit scores.

The median credit score for approved loans is around 700–750. For non-approved loans, the median is around 450–500. Therefore, Credit score appears to be a key predictor.

The histrogram suggests credit scores below 600 are more likely to be rejected, while above 650 are mostly approved.

### Income distribution by loan approved

The boxplot shows approved applicants generally have higher incomes.

The median income for approved loans is around 100,000, compared to about 80,000 for non-approved.

Income influences approval, but the difference is less pronounced than for credit score or points.

The histrogram suggests higher income levels correspond to higher loan approval rates. Applicants earning below ~70k have more rejections, while those above ~90k tend to get approved.

### Loan amount distribution by loan approved

The bocplot shows loan amounts are quite similar between approved and non-approved groups.

The median loan amount for approved loans is slightly lower than for rejected loans. Therefore, applicants asking for larger loans may be more likely to be rejected.

The histogram suggests applicants requesting larger loans are less likely to be approved, while those asking for smaller to moderate loan amounts have a higher chance of approval.

### Years employed distribution by loan approved

The boxplot shows years employed appears to have a positive relationship with loan approval. However, the median difference is small, and there’s overlap between the groups.

This suggests that while employment stability helps, it’s not the strongest predictor.

The histogram suggests applicants with longer employment histories are slightly more likely to be approved, but there are still many rejections across all experience levels.

### Loan to income distribution by loan approved

The boxplot and histogram suggests that as loan amount becomes large relative to income, the likelihood of approval descreases. However, applicants with a lower loan to income ratio are more likely to have their loans approved.

Applicants requesting loans close to or exceeding their income level (outliers) are often rejected.
The presence of numerous outliers in the rejected group indicates riskier financial profiles, which lenders are less likely to approve.

Outliners - https://www.youtube.com/shorts/SH7TPbT6zqE this link was used to help code the IRQ 

In [ ]:
outliers_summary = {}

for col in ['credit_score', 'points', 'loan_to_income']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3-Q1

    upper_bound = Q3 + 1.5 * IQR
    lower_bound = Q1 - 1.5 * IQR

    mask = (df[col] < lower_bound) | (df[col] > upper_bound)
    outliers = df[mask]

    outliers_summary[col] = {
        'count': mask.sum(),
        'outliers': outliers.copy()
    }

    print(f"\n{col}:")
    print(f"  Number of outliers: {mask.sum()}")

The outliers summary identified that the variable loan_to_income has 115 outliers.

In [ ]:
percent = (mask.sum() / len(df) * 100).round(2)

In [ ]:
print(f"{col}: {mask.sum()} outliers ({percent}%)")

The outliers consist of 5.75% of the dataset for the variable loan_to_income, which is significant. 

In [ ]:
sns.heatmap(df_eda.corr(numeric_only=True), annot=True, cmap='coolwarm')


Discretize credit_score and points and Parallel plot

In [ ]:
credit_map = [-np.inf, 580, 670, 740, 800, np.inf]
points_map = [-np.inf, 40, 60, 80, np.inf]
lti_map = [-np.inf, 0.3, 0.5, 0.7, 1.0, np.inf]

disc = ArbitraryDiscretiser(binning_dict={
    'credit_score': credit_map,
    'points': points_map,
    'loan_to_income': lti_map
})

df_eda['loan_to_income'] = df_eda['loan_amount'] / df_eda['income']
df_disc = disc.fit_transform(df_eda.copy())

In [ ]:
def make_label_map(binner_dict, variable):
    bins = binner_dict[variable]
    df_disc['loan_approved'] = df_disc['loan_approved'].replace({'No': '0', 'Yes': '1'})
    n_classes = len(bins) - 1
    classes_ranges = bins[1:-1]
    labels_map = {}
    for n in range(n_classes):
        if n == 0:
            labels_map[n] = f"<{classes_ranges[0]}"
        elif n == n_classes - 1:
            labels_map[n] = f"+{classes_ranges[-1]}"
        else:
            labels_map[n] = f"{classes_ranges[n-1]} to {classes_ranges[n]}"
    return labels_map

for var in ['credit_score', 'points', 'loan_to_income']:
    if var in disc.binner_dict_:
        df_disc[var] = df_disc[var].replace(make_label_map(disc.binner_dict_, var))
    else:
        print(f"Skipping {var} - not found in discretizer")


In [ ]:
for col in ['credit_score', 'points', 'loan_to_income']:
    df_disc[col] = df_disc[col].astype(str)

df_disc['loan_approved'] = df_disc['loan_approved'].astype(int)

fig = px.parallel_categories(
    df_disc[['credit_score', 'points', 'loan_to_income', 'loan_approved']],
    color="loan_approved",
    color_continuous_scale=px.colors.sequential.Plasma
)

fig.show()





The plot suggests:

Credit score and points are both positively correlated with loan approval.

Applicants with low credit score and low points have the highest rejection rates.

Applicants with high credit score and high points almost always get approved.

Applicants with lower loan-to-income ratios—specifically below 0.5 show a much higher proportion of loan approvals, indicating that lenders favor borrowers whose loan amount is small relative to their income. As the loan-to-income ratio increases beyond 0.7, there is a higher rate of loan rejections.

In [ ]:
ranking = pd.DataFrame({
    'Feature': corr_pearson.index,
    'Pearson_Corr': corr_pearson.values,
    'Spearman_Corr': corr_spearman.values
})
top_5 = ranking.head(5)[['Feature', 'Pearson_Corr']]
top_5


## Conclusions and Next Steps

* Applicants with high points are more likely to get approved.

* Applicants with high credit scores are more likely to get approved.

* Applicants with low points and low credit scores are more likely to be rejected.

* Income, loan amount, and years employed have minor effects on approval.

* City has almost no effect on approval.
  
* The parallel categories plot highlights a strong relationship between the loan-to-income ratio and loan approval outcomes.
  
* The best features to analyse include points, credit score and loan to income.

Next steps:

* Use points, credit score and loan to income as primary features in predictive models.
* Consider using a RobustScaler when preparing for linear regression rather than StandardScaler or MinMax Scale due to significant number of outliers in the loan_to_amount variable.